<a href="https://colab.research.google.com/github/mshruthi-cmi/Fashion-MNIST-Classification/blob/main/fashion_mnist_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

In [4]:
from google.colab import files
uploaded = files.upload()

Saving fashion-mnist_train.csv to fashion-mnist_train.csv


In [5]:
from google.colab import files
uploaded = files.upload()

Saving fashion-mnist_test.csv to fashion-mnist_test.csv


In [6]:
# Load Fashion-MNIST dataset
train = pd.read_csv('fashion-mnist_train.csv')
test = pd.read_csv('fashion-mnist_test.csv')

In [7]:
# Explore the data structure
train.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Split features and target variables
x_train = train.drop(columns='label', axis=1)
y_train = train['label']
x_test = test.drop(columns='label', axis=1)
y_test = test['label']

In [9]:
# Initialize PCA to retain 90% of variance
pca = PCA(n_components=0.9)

In [10]:
# Define models and their hyperparameter grids
models = {
   'LogisticRegression': {
       'model': LogisticRegression(),
       'param': {
           'clf__C': [0.1],
           'clf__penalty': ['l2'],
           'clf__solver': ['saga'],
           'clf__max_iter': [100000]
       }
   },
   'RandomForestClassifier': {
       'model': RandomForestClassifier(),
       'param': {
           'clf__max_depth': [19, 20, 21],
           'clf__criterion': ['gini', 'entropy']
       }
   },
   'KNeighborsClassifier': {
       'model': KNeighborsClassifier(),
       'param': {
           'clf__n_neighbors': [6, 7, 8],
           'clf__weights': ['distance', 'uniform']
       }
   }
}

In [11]:
# Store grid search results
grid_scores = {}

In [12]:
# Perform grid search for each model
for name, model in models.items():
   # Create pipeline with PCA and classifier
   pipe = Pipeline(steps=[
       ('pca', pca),
       ('clf', model['model'])
   ])

   # Setup grid search with cross-validation
   grid = GridSearchCV(
       pipe,
       param_grid=model['param'],
       cv=3,
       scoring='accuracy',
       n_jobs=-1,
       verbose=1
   )

   # Train the model
   grid.fit(x_train, y_train)

   # Store results
   grid_scores[name] = {
       'best_score': grid.best_score_,
       'best_params': grid.best_params_,
       'grid_object': grid
   }

   print(f"{name} - Best CV Score: {grid.best_score_}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
LogisticRegression - Best CV Score: 0.8427333333333333
Fitting 3 folds for each of 6 candidates, totalling 18 fits
RandomForestClassifier - Best CV Score: 0.8653333333333334
Fitting 3 folds for each of 6 candidates, totalling 18 fits
KNeighborsClassifier - Best CV Score: 0.8616999999999999


In [13]:
# Display best parameters for each model
for name, result in grid_scores.items():
   print(f"\n{name}:")
   print(f"Score: {result['best_score']:.4f}")
   print("Parameters:")
   for param, value in result['best_params'].items():
       print(f"  {param}: {value}")


LogisticRegression:
Score: 0.8427
Parameters:
  clf__C: 0.1
  clf__max_iter: 100000
  clf__penalty: l2
  clf__solver: saga

RandomForestClassifier:
Score: 0.8653
Parameters:
  clf__criterion: entropy
  clf__max_depth: 19

KNeighborsClassifier:
Score: 0.8617
Parameters:
  clf__n_neighbors: 6
  clf__weights: distance


In [14]:
# Evaluate models on test set
for name, result in grid_scores.items():
   # Get the best trained model
   best_model = result['grid_object'].best_estimator_

   # Make predictions on test set and check accuracy
   y_pred = best_model.predict(x_test)

   # Display results
   print(f"\n{'='*20} {name} {'='*20}")
   print(f"Test Accuracy: {best_model.score(x_test, y_test):.4f}")
   print("\nClassification Report:")
   print(classification_report(y_test, y_pred))


==================== LogisticRegression ====================
Test Accuracy: 0.8490

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.81      0.80      1000
           1       0.96      0.97      0.96      1000
           2       0.76      0.75      0.76      1000
           3       0.85      0.87      0.86      1000
           4       0.77      0.80      0.78      1000
           5       0.92      0.92      0.92      1000
           6       0.65      0.58      0.61      1000
           7       0.90      0.91      0.90      1000
           8       0.95      0.94      0.94      1000
           9       0.93      0.94      0.93      1000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


==================== RandomForestClassifier ====================
Test Accuracy: 0.8672

Classification Report:
              precisi